## Steps to Get Data from Redshift to Local and train local models

1. **Connect to VPN**  
   Connect to: `vpn.ao.zapsi.net`

2. **Access Redshift**  
   Log in to AWS and open the Redshift Query Editor:  
   [https://af-south-1.console.aws.amazon.com/sqlworkbench/home?region=af-south-1#/client](https://af-south-1.console.aws.amazon.com/sqlworkbench/home?region=af-south-1#/client)

3. **Check Connection Settings**  
   Edit the **DEV** connection if needed.
   Create a `.env` file with the REDSHIFT_PASSWORD variable, like:
   ```bash
   REDSHIFT_PASSWORD=R...%.
   ```

4. **Download Data**  
   Run this notebook to pull data from Redshift.  
   If required libraries are missing, install them:
   ```bash
   conda activate mlcourse
   conda install -c conda-forge python-dotenv
   conda install -c conda-forge pyarrow
   ````

    Then **restart the kernel**.

5. **Perform EDA**
Run cells from the `eda_example notebook` (run only from Section 7 onwards, if you want to jump to data_preparation notebook)

6. **Prepare Data for Training**
Run and adjust the `1_data_preparation_example` notebook to get the data ready for model training.


In [5]:
from dotenv import load_dotenv
import os
import psycopg2
import pandas as pd

# Load environment variables from .env file
load_dotenv()

# ====== CONFIG ======
REDSHIFT_HOST = "redshift-cluster-dsi.cl4o4mmtx9ir.af-south-1.redshift.amazonaws.com"
REDSHIFT_PORT = 5439
REDSHIFT_DB   = "dev"
REDSHIFT_USER = "awsuser"

# put your password in an env var before running:
# export REDSHIFT_PASSWORD='your-password-here'
REDSHIFT_PASSWORD = os.getenv('REDSHIFT_PASSWORD')

# Example query – change this to your table
QUERY = "SELECT * FROM home_awsidc_mario_negas_zap_co_ao_ed45de.df_final_ultimo_carregamento_mn_01 LIMIT 10000;"

OUTPUT_PARQUET = "../sample_data_from_redshift/sample.parquet"
# ====================


if not REDSHIFT_PASSWORD:
    raise RuntimeError("REDSHIFT_PASSWORD env var not set")

conn = psycopg2.connect(
    host=REDSHIFT_HOST,
    port=REDSHIFT_PORT,
    dbname=REDSHIFT_DB,
    user=REDSHIFT_USER,
    password=REDSHIFT_PASSWORD,
)

try:
    # read into pandas
    df = pd.read_sql(QUERY, conn)

    print(f"Fetched {len(df)} rows from Redshift")
    print(df.head())

    # save locally
    df.to_parquet(OUTPUT_PARQUET, index=False)
    print(f"Saved {len(df)} rows to {OUTPUT_PARQUET}")
finally:
    conn.close()



/var/folders/m4/1gplgrfj36v999v_z_yn3m1w0000gn/T/ipykernel_20240/1060326786.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(QUERY, conn)


Fetched 4787 rows from Redshift
   idconta codigoconta  iddim_date  carreg_iddim_date
0  3102206  1906918601    20251021           20250820
1  3910665  1314973002    20251025           20250818
2  3443234  1801871302    20251025           20250821
3  2822098  1001630703    20251023           20250821
4  4244607  2314021501    20251025           20250810
Saved 4787 rows to ../sample_data_from_redshift/sample.parquet


In [1]:
%pip install ydata-profiling

Note: you may need to restart the kernel to use updated packages.


In [3]:
from ydata_profiling import ProfileReport
profile = ProfileReport(df, title="EDA Report — df_final_ultimo_carregamento", explorative=True)
profile.to_file("eda_example_results/eda_report.html")

print("💡 Tip: Install ydata-profiling for automated EDA reports")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:00<00:00, 4060.31it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

💡 Tip: Install ydata-profiling for automated EDA reports
